# Ornstein-Ulenbeck, Black-Scholes, initiation `pytorch`

In [ ]:
import numpy as np
import scipy.stats as sps
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
from numpy.random import default_rng, SeedSequence

sq = SeedSequence()
rng = default_rng(sq);

## Ornstein-Uhlenbeck

On considère $(X_t)_{t \in [0,T]}$ processus à valeurs dans $\mathbf{R}$ solution de l'EDS 
\begin{equation*}
    \operatorname{d}\! X_t = \lambda (\mu - X_t) \operatorname{d}\! t + \sigma \operatorname{d}\! W_t, \quad X_0 = x \in \mathbf{R},
\end{equation*}
où $\lambda > 0$, $\mu, \sigma \in \mathbf{R}$.

On pose $Y_t = e^{\lambda t} (X_t - \mu)$ qui vérifie 
\begin{equation*}
\begin{aligned}
    \operatorname{d}\! Y_t &= \lambda e^{\lambda t} (X_t - \mu) \operatorname{d}\! t + e^{\lambda t} \operatorname{d}\! X_t, \\
    &= \sigma e^{\lambda t} \operatorname{d}\! W_t,
\end{aligned}
\end{equation*}
et $Y_0 = x-\mu$. 

Le processus $(Y_t)_{t \in [0,T]}$ est une intégrale de Wiener, donc un processus de Markov et un processus Gaussien de fonction moyenne $m_Y(t) = x-\mu$ et de fonction de covariance 
\begin{equation*}
\begin{aligned}
    K_Y(s, t) &= \operatorname{cov}(Y_s, Y_t) = \sigma^2 \mathbf{E}\bigg[\int_0^s e^{\lambda u} \operatorname{d}\! W_u \int_0^t e^{\lambda u} \operatorname{d}\! W_u \bigg], \\
            &= \frac{\sigma^2}{2 \lambda} \big( e^{2 \lambda \min(s,t)} - 1 \big).
\end{aligned}
\end{equation*}
En particulier, 
\begin{equation*}
    \forall 0 \le s \le t, \quad \mathcal{L}\big(Y_t| \sigma(Y_u, u \le s)\big) 
    \sim \mathcal{N}\Bigl( Y_s ; \frac{\sigma^2}{2 \lambda} \bigl( e^{2 \lambda t} - e^{2 \lambda s}\bigr) \Bigr) 
\end{equation*}

On en déduit que $(X_t)_{t \in [0,T]}$ un aussi un processus Gaussien de fonction moyenne $m_X$ et de fonction de covariance $K_X$ définies par 
\begin{equation*}
    m_X(t) = \mu + (x-\mu) e^{-\lambda t} \quad \text{et} \quad 
    K_X(s, t) = \frac{\sigma^2}{2 \lambda} e^{-\lambda (s+t)} \big( e^{2 \lambda \min(s,t)} - 1 \big).
\end{equation*}
En particulier, 
\begin{equation*}
    \forall 0 \le s \le t, \quad \mathcal{L}\big(X_t| \sigma(X_u, u \le s)\big) 
    \simeq \mathcal{N}\Bigl( \mu + (X_s-\mu) e^{-\lambda(t-s)} ; \frac{\sigma^2}{2 \lambda} \bigl( 1 - e^{-2\lambda(t-s)} \bigr) \Bigr) 
\end{equation*}

In [ ]:
lambd = 5
mu = 1.
sigma = 0.3
x0 = 3.0
T = 5

Au temps $T = 5$ on a $X_T$ qui suit une loi Normale de paramètres:

In [ ]:
mean = mu + (x0-mu)*np.exp(-lambd*T)
var = sigma**2/(2*lambd)*(1-np.exp(-2*lambd*T))
print("mean:\t", mean)
print("var:\t", var)

### Question: Simulation via $Y$

Ecrire une fonction 
```
    ou_direct_np(noise, delta, x0, lambd, mu, sigma)
```
qui prend en argument `noise` un `np.array` de shape `(n, M)` où `n` est le nombre de pas de temps et `M` le nombre de trajectoires à simuler, `delta` est $\delta = T/n$ et `x0`, `lambd`, `mu` et `sigma` sont les paramètres du processus d'Ornstein-Uhlenbeck.

L'implémentation doit être la suivante: on simule les accroissements de $(Y_t)_{t \in [0,T]}$ aux instants $t_k = k \delta$, puis on reconstruit les $Y_{t_k}$ (via `np.cumsum`) et on obtient $X_{t_k}$ à partir de $Y_{t_k}$.

### Question: Simulation récursive

De façon similaire, écrire une fonction 
```
    ou_rec_np(noise, delta, x0, lambd, mu, sigma)
```
qui utilise une boucle `for` pour construire itérativement $X_{t_{k+1}}$ à partir de $X_{t_k}$ (les $M$ trajectoires en même temps).

### Question: vérification code et visualisation

Vérifier en utilisant le même argument `noise` que les deux fonctions renvoient les mêmes trajectoires. Vérifier aussi la moyenne empirique et la variance empirique terminale.

Tracer 100 trajectoires.

### Question: comparaison des temps 

Comparer les temps de calcul pour différentes valeurs de `n` et `M`.

## Algorithmes en `pytorch`

Reprendre les deux fonctions précédentes pour adapter le code à `pytorch`: on remplace les `np.array` de `numpy` par des `torch.tensor` de `pytorch`.

[Documentation officielle à lire.](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)

### Question: Implémentation sur CPU

Ecrire des fonctions `ou_direct_torch` et `ou_rec_torch` pour utiliser `pytorch` à la place de `numpy`.

Il suffit d'importer le module `torch` et de remplacer les appels de `numpy` par les fonctions/objets équivalents en pytorch. 

### Question: Tester et comparer les temps 

Vérifier vos codes en calculant moyenne et variance sur l'échantillon au temps final. 

Mesurer les temps de calculs et les comparer avec `numpy`.

### Question: Implémentation sur un autre device

Pour utiliser un autre device, il faut modifier son code pour demander la création des tensors sur le device en question. 

Adapter le code précédent en ajoutant l'option `device` aux fonctions `ou_direct_torch` et `ou_rec_torch`.

## Brownien géométrique, modèle de Black-Scholes

On considère $(S_t)_{t \in [0,T]}$ solution de l'EDS sur $[0,T]$
\begin{equation*}
    \operatorname{d}\!S_t = r S_t \operatorname{d}\!t + \sigma S_t \operatorname{d}\!B_t, \quad S_0 = x
\end{equation*}
c'est à dire 
\begin{equation*}
    S_t = x \exp \bigl((r- \sigma^2/2) t + \sigma B_t \bigr).
\end{equation*}

### Question: Simulation directe

Réfléchir à deux algorithmes différents: l'un en utilisant la transformation exponentielle appliquée à un Brownien (dilaté et translaté), l'autre en utilisant un produit cumulé `cumprod`.

Ecrire un code de simulation `numpy` puis `pytorch` pour simuler `M` trajectoires aux instants $t_k = k T/n$, $n \ge 1$.